## Imports

In [ ]:
from pathlib import Path

import tiktoken
import torch
import torch.nn as nn

from nano_llm.loaders import create_dataloader_v1

## Variables

In [ ]:
TEXT_PATH = Path("..") / "the-verdict.txt"
VOCAB_SIZE = tiktoken.get_encoding("gpt2").n_vocab
CONTEXT_LENGTH = 4
EMBEDDING_DIM = 256
DROPOUT_RATE = 0.1

raw_text = TEXT_PATH.read_text(encoding="utf-8")

## Loader

In [ ]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=2, stride=2, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
second_batch = next(data_iter)

print(first_batch)
print("====")
print(second_batch)

## Token embeddings

In [ ]:
torch.manual_seed(123)

dataloader = create_dataloader_v1(
    raw_text,
    batch_size=8,
    max_length=CONTEXT_LENGTH,
    stride=CONTEXT_LENGTH,
    shuffle=False,
)
inputs, targets = next(iter(dataloader))

token_embedding_layer = torch.nn.Embedding(VOCAB_SIZE, EMBEDDING_DIM)
token_embeddings = token_embedding_layer(inputs)

print("Token IDs:\n", inputs)
print("Token embeddings shape:", token_embeddings.shape)

## Positional embeddings

In [ ]:
pos_embedding_layer = torch.nn.Embedding(CONTEXT_LENGTH, EMBEDDING_DIM)
pos_embeddings = pos_embedding_layer(torch.arange(CONTEXT_LENGTH))

input_embeddings = token_embeddings + pos_embeddings

print("Positional embeddings shape:", pos_embeddings.shape)
print("Input embeddings shape:", input_embeddings.shape)

## Self-Attention

In [ ]:
class SelfAttentionV2(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(context_length, context_length, dtype=torch.bool),
                diagonal=1,
            ),
        )

    def forward(self, x):
        num_tokens = x.shape[1]
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.mT
        attn_scores = attn_scores.masked_fill(
            self.mask[:num_tokens, :num_tokens], -torch.inf
        )
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vec = attn_weights @ values
        return context_vec

In [ ]:
torch.manual_seed(123)

self_attention_layer = SelfAttentionV2(
    EMBEDDING_DIM, EMBEDDING_DIM, CONTEXT_LENGTH, DROPOUT_RATE
)
context_vectors = self_attention_layer(input_embeddings)

print("Input embeddings shape:", input_embeddings.shape)
print("Context vectors shape:", context_vectors.shape)